In [22]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import json
from urllib.parse import urlparse
import re
from tqdm import tqdm

In [23]:
# Load your existing dataset
df = pd.read_csv('data/processed/ted_talks_all_clean.csv', nrows=10)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()
df_test = df.head(10)  # Test with first 10 rows

Dataset shape: (10, 4)
Columns: ['title', 'speaker', 'duration', 'url']


In [47]:
def extract_transcript(soup, headers):
    """Extraire le transcript depuis la page (version améliorée)"""
    try:
        # Méthode 1: Chercher dans JSON-LD (schema.org)
        script_tag = soup.find('script', type='application/ld+json')
        if script_tag:
            try:
                ld_data = json.loads(script_tag.string)
                transcript = ld_data.get('transcript', '')
                if transcript and len(transcript) > 100:
                    print(f"✓ Transcript trouvé dans JSON-LD ({len(transcript)} chars)")
                    return transcript
            except Exception as e:
                print(f"Erreur JSON-LD: {e}")

        # Méthode 2: Chercher dans __NEXT_DATA__
        nextjs_data = soup.find('script', id='__NEXT_DATA__')
        if nextjs_data:
            try:
                data = json.loads(nextjs_data.string)
                # Chercher récursivement le transcript
                transcript = find_transcript_in_json(data)
                if transcript:
                    print(f"✓ Transcript trouvé dans __NEXT_DATA__ ({len(transcript)} chars)")
                    return transcript
            except Exception as e:
                print(f"Erreur __NEXT_DATA__: {e}")

        # Méthode 3: Debug - afficher ce qu'on trouve
        print("Aucun transcript trouvé. Debug info:")
        print(f"  - Scripts JSON-LD: {bool(script_tag)}")
        print(f"  - Scripts __NEXT_DATA__: {bool(nextjs_data)}")
        
        # Afficher les clés disponibles dans __NEXT_DATA__ pour debug
        if nextjs_data:
            try:
                data = json.loads(nextjs_data.string)
                talk_data = data.get('props', {}).get('pageProps', {}).get('videoData', {})
                print(f"  - Clés dans videoData: {list(talk_data.keys())}")
            except:
                pass
        
        return ""
        
    except Exception as e:
        print(f"Error extracting transcript: {e}")
        return ""

def find_transcript_in_json(data):
    """Chercher récursivement le transcript dans la structure JSON"""
    if isinstance(data, dict):
        # Chercher dans les clés possibles pour le transcript
        for key in ['transcript', 'transcriptText', 'fullTranscript']:
            if key in data and data[key]:
                return data[key]
        
        # Chercher récursivement dans les valeurs
        for value in data.values():
            result = find_transcript_in_json(value)
            if result:
                return result
                
    elif isinstance(data, list):
        for item in data:
            result = find_transcript_in_json(item)
            if result:
                return result
    return None

def scrape_single_ted_talk(url, headers):
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        nextjs_data = soup.find('script', id='__NEXT_DATA__')
        if not nextjs_data:
            return None
            
        data = json.loads(nextjs_data.string)
        talk_data = data.get('props', {}).get('pageProps', {}).get('videoData', {})
        
        if not talk_data:
            talk_data = data.get('props', {}).get('pageProps', {})

        # APPEL DE LA FONCTION EXTRACT_TRANSCRIPT
        transcript = extract_transcript(soup, headers)
        
        # Extract topics
        topics = talk_data.get('topics', {})
        topic_nodes = topics.get('nodes', []) if isinstance(topics, dict) else topics
        topic_names = [t.get('name') for t in topic_nodes if isinstance(t, dict)]
        
        # Extract ratings (Inspiring, Informative, etc.)
        ratings = talk_data.get('ratings', [])
        rating_dict = {r.get('name'): r.get('count') for r in ratings if isinstance(r, dict)}

        return {
            'id': talk_data.get('id'),
            'title': talk_data.get('title'),
            'speaker': talk_data.get('presenterDisplayName'),
            'description': talk_data.get('description'),
            'recorded_at': talk_data.get('recordedAt'),
            'duration': talk_data.get('duration'),
            'duration_min': talk_data.get('duration', 0) // 60,
            'views': talk_data.get('viewedCount'),
            'topics': ', '.join(topic_names),
            'num_topics': len(topic_names),
            'video_context': talk_data.get('videoContext'),
            'type': talk_data.get('type', {}).get('name') if isinstance(talk_data.get('type'), dict) else None,
            'language': talk_data.get('language'),
            'num_subtitles': len(talk_data.get('translations', [])),
            'url': talk_data.get('canonicalUrl'),
            
            # Engagement metrics
            'tedcom_percentage': talk_data.get('tedcomPercentage'),
            'youtube_percentage': talk_data.get('youtubePercentage'),
            'podcasts_percentage': talk_data.get('podcastsPercentage'),
            
            # NOUVEAU: Transcript
            'transcript': transcript,
            'transcript_length': len(transcript) if transcript else 0,
        }
    
    except Exception as e:
        print(f"Error: {str(e)[:50]}")
        return None

In [48]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

results = []
for idx, row in df_test.iterrows():
    print(f"{idx+1}/{len(df_test)}: {row['title'][:50]}...", end=" ")
    
    extra_data = scrape_single_ted_talk(row['url'], headers)
    
    combined = {
        'title': row['title'],
        'speaker': row['speaker'],
        'duration': row['duration'],
        'url': row['url'],
    }
    if extra_data:
        combined.update(extra_data)
        print("✓")
        if extra_data.get('transcript'):
            print(f"  Transcript: {len(extra_data['transcript'])} chars")
    else:
        print("✗")
    
    results.append(combined)
    time.sleep(1)

df_result = pd.DataFrame(results)
print(f"\nScraped {df_result['id'].notna().sum()}/{len(df_result)} successfully")
df_result

1/10: A pastry chef works his chocolatier magic \'97 liv... ✓ Transcript trouvé dans JSON-LD (10422 chars)
✓
  Transcript: 10422 chars
2/10: The flourishing future of women's sports... ✓ Transcript trouvé dans JSON-LD (11310 chars)
✓
  Transcript: 11310 chars
3/10: How we\'92re turning pollution into toys, toothpas... ✓ Transcript trouvé dans JSON-LD (10275 chars)
✓
  Transcript: 10275 chars
4/10: The best thing that could happen to the energy ind... ✓ Transcript trouvé dans JSON-LD (9406 chars)
✓
  Transcript: 9406 chars
5/10: 3 simple ways to build stronger relationships at w... ✓ Transcript trouvé dans JSON-LD (13084 chars)
✓
  Transcript: 13084 chars
6/10: How video games can power up your parenting... ✓ Transcript trouvé dans JSON-LD (12660 chars)
✓
  Transcript: 12660 chars
7/10: Why we need to know our lives matter... ✓ Transcript trouvé dans JSON-LD (9069 chars)
✓
  Transcript: 9069 chars
8/10: How nearly dying helped me discover my own cure (a... ✓ Transcript trouvé dans JSON-

,title,speaker,duration,url,id,description,recorded_at,duration_min,views,topics,num_topics,video_context,type,language,num_subtitles,tedcom_percentage,youtube_percentage,podcasts_percentage,transcript,transcript_length
0,A pastry chef works his chocolatier magic — live,Amaury Guichon,757,https://www.ted.com/talks/amaury_guichon_a_pas...,155034,Get a taste of the chocolatier life from world...,None,12,237451,"design, innovation, food, creativity, art",5,TED2025,TED Stage Talk,en,0,0.051,0.0,0.931,"Latif Nasser: OK, chef, I think it&apos;s safe...",10422
1,The flourishing future of women's sports,Kate Johnson,772,https://www.ted.com/talks/kate_johnson_the_flo...,162374,Women's sports are surging in popularity aroun...,None,12,232089,"culture, technology, media, sports, AI, algorithm",6,TEDSports Indianapolis 2025,TED Stage Talk,en,0,0.034,0.0,0.954,The Olympics has always served as a touch poin...,11310
2,"How we’re turning pollution into toys, toothpa...",Xu Hao,781,https://www.ted.com/talks/xu_hao_how_we_re_tur...,160684,It took alcohol 200 years to go from scientifi...,None,13,236452,"climate change, science, sustainability, techn...",8,TED Countdown Summit 2025,TED Stage Talk,en,0,0.039,0.0,0.944,Humans have been using alcohol for thousands o...,10275
3,The best thing that could happen to the energy...,Matt Tilleard,769,https://www.ted.com/talks/matt_tilleard_the_be...,161395,History has been written by whoever controls t...,None,12,257728,"climate change, politics, sustainability, ener...",9,TED Countdown Summit 2025,TED Stage Talk,en,0,0.049,0.0,0.908,Our modern world was built on fuel. History ha...,9406
4,3 simple ways to build stronger relationships ...,Alyssa Birnbaum,903,https://www.ted.com/talks/alyssa_birnbaum_3_si...,161270,Doing the best at your job isn't just about wo...,None,15,311915,"business, psychology, relationships, communica...",6,TEDxClaremontGraduateUniversity,TEDx Talk,en,0,0.15,0.0,0.806,I remember the stomach flutters before my firs...,13084
5,How video games can power up your parenting,Hannah Boquet,856,https://www.ted.com/talks/hannah_boquet_how_vi...,160410,Parenting an eye-rolling teenager glued to a g...,None,14,263358,"technology, entertainment, relationships, pare...",9,TEDxSioux Falls,TED Stage Talk,en,0,0.079,0.0,0.888,"Before I get started, I just want a quick show...",12660
6,Why we need to know our lives matter,Jennifer Wallace,751,https://www.ted.com/talks/jennifer_wallace_why...,148433,It’s not enough to do important work — we need...,None,12,325695,"culture, community, work, personal growth, soc...",6,TED2025,TED Stage Talk,en,0,0.064,0.116,0.795,When we think about the most meaningful jobs i...,9069
7,How nearly dying helped me discover my own cur...,David Fajgenbaum,840,https://www.ted.com/talks/david_fajgenbaum_how...,157718,Physician-scientist David Fajgenbaum was dying...,None,14,408836,"science, technology, disease, health, health c...",8,TED2025,TED Stage Talk,en,0,0.172,0.208,0.597,"Hi, I&apos;m David Fajgenbaum and this is me i...",14395
8,Could we detect breast cancer with a fingerprint?,Simona Francese,749,https://www.ted.com/talks/simona_francese_coul...,159724,Breast cancer is the most common cancer among ...,None,12,286801,"science, technology, health, health care, canc...",9,TEDxManchester,TEDx Talk,en,0,0.07,0.0,0.91,"In this room, one in eight women will develop ...",9887
9,Why you should spend less time with your kids,Lenore Skenazy,794,https://www.ted.com/talks/lenore_skenazy_why_y...,153162,"Whether it’s micromanaging playtime, constantl...",None,13,480262,"education, social change, parenting, personal ...",6,TED2025,TED Stage Talk,en,0,0.147,0.248,0.561,"I am here to talk about parenting, which is ki...",11707


In [30]:
def scrape_single_ted_talk(url, headers):
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        nextjs_data = soup.find('script', id='__NEXT_DATA__')
        if not nextjs_data:
            return None
            
        data = json.loads(nextjs_data.string)
        talk_data = data.get('props', {}).get('pageProps', {}).get('videoData', {})
        
        if not talk_data:
            talk_data = data.get('props', {}).get('pageProps', {})

        print(f"\nAvailable keys in talk_data:")
        print(list(talk_data.keys()))
    
    
    except Exception as e:
        print(f"Error: {str(e)[:50]}")
        return None

In [31]:
scrape_single_ted_talk('https://www.ted.com/talks/sir_ken_robinson_do_schools_kill_creativity', headers)


Available keys in talk_data:
['__typename', 'playerData', 'takeaways', 'talkExtras', 'relatedVideos', 'speakers', 'type', 'description', 'socialTitle', 'internalLanguageCode', 'commentsEnabled', 'commentsLoggedInOnly', 'recordedOn', 'curatorApproved', 'socialDescription', 'partnerName', 'videoContext', 'audioInternalLanguageCode', 'language', 'hasTranslations', 'featured', 'customPartnerContent', 'topics', 'presenterDisplayName', 'duration', 'canonicalUrl', 'viewedCount', 'tedcomPercentage', 'youtubePercentage', 'podcastsPercentage', 'tedappsPercentage', 'publishedAt', 'id', 'title', 'slug', 'primaryImageSet']


In [32]:
def scrape_single_ted_talk(url, headers):
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        nextjs_data = soup.find('script', id='__NEXT_DATA__')
        if not nextjs_data:
            return None
            
        data = json.loads(nextjs_data.string)
        talk_data = data.get('props', {}).get('pageProps', {}).get('videoData', {})
        
        if not talk_data:
            talk_data = data.get('props', {}).get('pageProps', {})

        print(f"\nAvailable keys in talk_data:")
        print(list(talk_data.keys())[:20])
        
        # Extract topics
        topics = talk_data.get('topics', {})
        topic_nodes = topics.get('nodes', []) if isinstance(topics, dict) else topics
        topic_names = [t.get('name') for t in topic_nodes if isinstance(t, dict)]
        
        # Extract ratings (Inspiring, Informative, etc.)
        ratings = talk_data.get('ratings', [])
        rating_dict = {r.get('name'): r.get('count') for r in ratings if isinstance(r, dict)}

        return {
            'id': talk_data.get('id'),
            'title': talk_data.get('title'),
            'speaker': talk_data.get('presenterDisplayName'),
            'description': talk_data.get('description'),
            'recorded_at': talk_data.get('recordedAt'),  # Event date
            'duration': talk_data.get('duration'),
            'duration_min': talk_data.get('duration', 0) // 60,  # Convert to minutes
            'views': talk_data.get('viewedCount'),
            'topics': ', '.join(topic_names),
            'num_topics': len(topic_names),
            'video_context': talk_data.get('videoContext'),  # Event name
            'type': talk_data.get('type', {}).get('name') if isinstance(talk_data.get('type'), dict) else None,
            'language': talk_data.get('language'),
            'num_subtitles': len(talk_data.get('translations', [])),  # Available languages
            'url': talk_data.get('canonicalUrl'),
            
            # Engagement metrics
            'tedcom_percentage': talk_data.get('tedcomPercentage'),
            'youtube_percentage': talk_data.get('youtubePercentage'),
            'podcasts_percentage': talk_data.get('podcastsPercentage'),
        }
    
    
    except Exception as e:
        print(f"Error: {str(e)[:50]}")
        return None
    


In [33]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

results = []
for idx, row in df_test.iterrows():
    print(f"{idx+1}/{len(df_test)}: {row['title'][:50]}...", end=" ")
    
    extra_data = scrape_single_ted_talk(row['url'], headers)
    
    combined = {
        'title': row['title'],
        'speaker': row['speaker'],
        'duration': row['duration'],
        'url': row['url'],
    }
    if extra_data:
        combined.update(extra_data)
        print("✓")
    else:
        print("✗")
    
    results.append(combined)
    time.sleep(1)

df_result = pd.DataFrame(results)
print(f"\nScraped {df_result['id'].notna().sum()}/{len(df_result)} successfully")
df_result

1/10: A pastry chef works his chocolatier magic \'97 liv... 
Available keys in talk_data:
['__typename', 'playerData', 'takeaways', 'talkExtras', 'relatedVideos', 'speakers', 'type', 'description', 'socialTitle', 'internalLanguageCode', 'commentsEnabled', 'commentsLoggedInOnly', 'recordedOn', 'curatorApproved', 'socialDescription', 'partnerName', 'videoContext', 'audioInternalLanguageCode', 'language', 'hasTranslations']
✓
2/10: The flourishing future of women's sports... 
Available keys in talk_data:
['__typename', 'playerData', 'takeaways', 'talkExtras', 'relatedVideos', 'speakers', 'type', 'description', 'socialTitle', 'internalLanguageCode', 'commentsEnabled', 'commentsLoggedInOnly', 'recordedOn', 'curatorApproved', 'socialDescription', 'partnerName', 'videoContext', 'audioInternalLanguageCode', 'language', 'hasTranslations']
✓
3/10: How we\'92re turning pollution into toys, toothpas... 
Available keys in talk_data:
['__typename', 'playerData', 'takeaways', 'talkExtras', 'relatedVi

,title,speaker,duration,url,id,description,recorded_at,duration_min,views,topics,num_topics,video_context,type,language,num_subtitles,tedcom_percentage,youtube_percentage,podcasts_percentage
0,A pastry chef works his chocolatier magic — live,Amaury Guichon,757,https://www.ted.com/talks/amaury_guichon_a_pas...,155034,Get a taste of the chocolatier life from world...,None,12,237451,"design, innovation, food, creativity, art",5,TED2025,TED Stage Talk,en,0,0.051,0.0,0.931
1,The flourishing future of women's sports,Kate Johnson,772,https://www.ted.com/talks/kate_johnson_the_flo...,162374,Women's sports are surging in popularity aroun...,None,12,232089,"culture, technology, media, sports, AI, algorithm",6,TEDSports Indianapolis 2025,TED Stage Talk,en,0,0.034,0.0,0.954
2,"How we’re turning pollution into toys, toothpa...",Xu Hao,781,https://www.ted.com/talks/xu_hao_how_we_re_tur...,160684,It took alcohol 200 years to go from scientifi...,None,13,236452,"climate change, science, sustainability, techn...",8,TED Countdown Summit 2025,TED Stage Talk,en,0,0.039,0.0,0.944
3,The best thing that could happen to the energy...,Matt Tilleard,769,https://www.ted.com/talks/matt_tilleard_the_be...,161395,History has been written by whoever controls t...,None,12,257728,"climate change, politics, sustainability, ener...",9,TED Countdown Summit 2025,TED Stage Talk,en,0,0.049,0.0,0.908
4,3 simple ways to build stronger relationships ...,Alyssa Birnbaum,903,https://www.ted.com/talks/alyssa_birnbaum_3_si...,161270,Doing the best at your job isn't just about wo...,None,15,311915,"business, psychology, relationships, communica...",6,TEDxClaremontGraduateUniversity,TEDx Talk,en,0,0.15,0.0,0.806
5,How video games can power up your parenting,Hannah Boquet,856,https://www.ted.com/talks/hannah_boquet_how_vi...,160410,Parenting an eye-rolling teenager glued to a g...,None,14,263358,"technology, entertainment, relationships, pare...",9,TEDxSioux Falls,TED Stage Talk,en,0,0.079,0.0,0.888
6,Why we need to know our lives matter,Jennifer Wallace,751,https://www.ted.com/talks/jennifer_wallace_why...,148433,It’s not enough to do important work — we need...,None,12,325695,"culture, community, work, personal growth, soc...",6,TED2025,TED Stage Talk,en,0,0.064,0.116,0.795
7,How nearly dying helped me discover my own cur...,David Fajgenbaum,840,https://www.ted.com/talks/david_fajgenbaum_how...,157718,Physician-scientist David Fajgenbaum was dying...,None,14,408836,"science, technology, disease, health, health c...",8,TED2025,TED Stage Talk,en,0,0.172,0.208,0.597
8,Could we detect breast cancer with a fingerprint?,Simona Francese,749,https://www.ted.com/talks/simona_francese_coul...,159724,Breast cancer is the most common cancer among ...,None,12,286801,"science, technology, health, health care, canc...",9,TEDxManchester,TEDx Talk,en,0,0.07,0.0,0.91
9,Why you should spend less time with your kids,Lenore Skenazy,794,https://www.ted.com/talks/lenore_skenazy_why_y...,153162,"Whether it’s micromanaging playtime, constantl...",None,13,480262,"education, social change, parenting, personal ...",6,TED2025,TED Stage Talk,en,0,0.147,0.248,0.561
